# Cria tabela fato fact_orders na camada gold

Essa é a primeira vez no projeto que combinamos duas tabelas numa só, é literalmente o que separa "dados limpos" (Silver) de "dados prontos pra análise de negócio" (Gold): a Gold junta as peças pra responder perguntas reais, tipo "quanto vendemos por categoria de produto no último trimestre".

In [0]:
%python
fact_orders = (
    spark.table("olist_project.silver.order_items")
    .join(
        spark.table("olist_project.silver.orders"),
        on="order_id",
        how="inner",
    )
    .select(
        "order_id",
        "order_item_id",
        "customer_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "order_purchase_timestamp",
        "order_status",
    )
)

(
    fact_orders.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.gold.fact_orders")
)

print(f"fact_orders: {fact_orders.count()} linhas")

In [0]:
DESCRIBE olist_project.gold.fact_orders

### query de validação

In [0]:
SELECT COUNT(*) as total_linhas,
       COUNT(DISTINCT order_id) as pedidos_unicos
FROM olist_project.gold.fact_orders

# Parte 2

# Dimensões: dim_customer, dim_product, dim_seller

## dim_customer

In [0]:
%python
dim_customer = (
    spark.table("olist_project.silver.customers")
    .select("customer_id", "customer_city", "customer_state")
)

(
    dim_customer.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.gold.dim_customer")
)

print(f"dim_customer: {dim_customer.count()} linhas")


## dim_product

In [0]:
%python
dim_product = (
    spark.table("olist_project.silver.products")
    .join(
        spark.table("olist_project.bronze.product_category_name_translation"),
        on="product_category_name",
        how="left",
    )
    .select("product_id", "product_category_name_english")
)

(
    dim_product.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.gold.dim_product")
)

print(f"dim_product: {dim_product.count()} linhas")


Traz a tradução da categoria (inglês) via JOIN com a tabela de tradução.

## dim_seller

In [0]:
%python
dim_seller = (
    spark.table("olist_project.silver.sellers")
    .select("seller_id", "seller_city", "seller_state")
)

(
    dim_seller.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.gold.dim_seller")
)

print(f"dim_seller: {dim_seller.count()} linhas")